# 🎙️ Snack4Pets — French UGC Voice Fitting (Fine-Tuning)

This notebook fine-tunes **F5-TTS** on custom French UGC creator audio (TikTok/Reels format) for **Snack4Pets**.

### Prerequisites:
- **Runtime:** GPU (T4 or higher). Check `Runtime -> Change runtime type -> T4 GPU`.
- **Audio dataset:** 3 to 10 minutes of clean, conversational French audio (from 1 creator speaking casually, phone mic or lapel mic style).
- Checkpoints can be saved directly to **Google Drive**.

## 1. Check GPU & System

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (Optional but Recommended)
Mount Google Drive to persist fine-tuned checkpoints and prevent losing models when Colab disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/snack4pets_tts_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {SAVE_DIR}")

## 3. Install Dependencies & Clone F5-TTS

In [ ]:
# Install ffmpeg and system deps
!apt-get -y update && apt-get install -y ffmpeg

# Clone and install F5-TTS
!git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS
%cd /content/F5-TTS
!pip install -e .
!pip install whisper jieba pypinyin datasets accelerate torchaudio soundfile librosa

## 4. Option A: Launch Interactive Fine-Tuning WebUI (Gradio)
The simplest method: Upload your `.wav` files via the Gradio interface, run automatic Whisper transcription in French, and start fine-tuning with a single click.

In [ ]:
%cd /content/F5-TTS
!python src/f5_tts/train/finetune_gradio.py --share

## 4. Option B: Programmatic Training Pipeline
If you prefer a headless automated workflow, upload your audio folder (`/content/raw_audio`) and run the cells below.

In [ ]:
import os
import glob
import whisper
import soundfile as sf
import librosa
import csv

AUDIO_DIR = '/content/raw_audio'
PROCESSED_DIR = '/content/processed_audio'
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Please upload your .wav / .mp3 clips to: {AUDIO_DIR}")

In [ ]:
# Auto-transcribe and format to metadata.csv
audio_files = glob.glob(f"{AUDIO_DIR}/*.*")

if len(audio_files) == 0:
    print("No audio files found yet. Upload clips into /content/raw_audio first.")
else:
    print(f"Transcribing {len(audio_files)} files using Whisper (French)...")
    model = whisper.load_model("base")
    metadata_path = '/content/metadata.csv'
    
    with open(metadata_path, 'w', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter='|')
        writer.writerow(['audio_file', 'text'])
        for idx, file_path in enumerate(audio_files):
            # Convert to 24kHz mono wav
            y, sr = librosa.load(file_path, sr=24000, mono=True)
            target_file = os.path.join(PROCESSED_DIR, f"sample_{idx:03d}.wav")
            sf.write(target_file, y, 24000)
            
            res = model.transcribe(target_file, language='fr')
            text = res['text'].strip()
            writer.writerow([target_file, text])
            print(f"Processed {target_file}: {text}")
            
    print(f"Metadata CSV written to {metadata_path}")

In [ ]:
# Launch Accelerate Training (Headless)
# 1000 - 3000 steps are usually sufficient for single-speaker fitting
%cd /content/F5-TTS
!python src/f5_tts/train/datasets/prepare_csv_wavs.py /content/metadata.csv /content/prepared_dataset
!accelerate launch --mixed_precision=fp16 src/f5_tts/train/train.py \
    --config-name F5TTS_v1_Base.yaml \
    ++datasets.batch_size_per_gpu=3840 \
    ++model.epochs=50

## 5. Copy Checkpoint to Google Drive

In [ ]:
# Copy latest model to Drive
!cp -r /content/F5-TTS/ckpts/* /content/drive/MyDrive/snack4pets_tts_checkpoints/
print("Checkpoints successfully archived to Google Drive.")